In [ ]:
# -*- coding: utf-8 -*-

import pandas as pd
import os
import psycopg2
from io import StringIO
from pathlib import Path
import logging

# ============================================================
# CONFIGURAÇÃO — use variáveis de ambiente antes de rodar
#
# Windows (PowerShell):
#   $env:DB_HOST="localhost"
#   $env:DB_NAME="dw_sorveteria"
#   $env:DB_USER="postgres"
#   $env:DB_PASSWORD="sua_senha"
#   $env:ARQUIVO_SORVETES="C:\caminho\base_sorvetes_alagoas_v4.csv"
#   $env:ARQUIVO_SUPERVISORES="C:\caminho\base_supervisores_metas_v2.csv"
#
# Linux / Mac:
#   export DB_PASSWORD="sua_senha"
#   export ARQUIVO_SORVETES="/caminho/base_sorvetes_alagoas_v4.csv"
# ============================================================

BATCH_SIZE = 50000

DB_CONFIG = {
    "host":     os.environ.get("DB_HOST",     "localhost"),
    "database": os.environ.get("DB_NAME",     "dw_sorveteria"),
    "user":     os.environ.get("DB_USER",     "postgres"),
    "password": os.environ.get("DB_PASSWORD", ""),
}

arquivo_sorvetes     = Path(os.environ.get("ARQUIVO_SORVETES",     "bases/base_sorvetes_alagoas_v4.csv"))
arquivo_supervisores = Path(os.environ.get("ARQUIVO_SUPERVISORES", "bases/base_supervisores_metas_v2.csv"))

os.makedirs("logs", exist_ok=True)

logging.basicConfig(
    filename="logs/etl_dw.log",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    encoding="utf-8"
)


def normalizar_colunas(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    return df


def limpar_textos(df):
    for col in df.columns:
        df[col] = df[col].astype(str).str.strip()
    return df


def tratar_numero_americano(serie):
    return pd.to_numeric(
        serie.astype(str)
        .str.strip()
        .str.replace("R$", "", regex=False)
        .str.replace(" ",  "", regex=False)
        .str.replace(",",  "", regex=False),
        errors="coerce"
    ).fillna(0)


def carregar_staging_em_batch(cursor, conn, df_staging):
    cursor.execute("TRUNCATE TABLE stg_vendas RESTART IDENTITY;")
    conn.commit()

    total = len(df_staging)

    for i in range(0, total, BATCH_SIZE):
        batch = df_staging.iloc[i:i + BATCH_SIZE]

        buffer = StringIO()
        batch.to_csv(
            buffer,
            index=False,
            sep=";",
            header=False,
            na_rep="",
            lineterminator="
"
        )
        buffer.seek(0)

        cursor.copy_expert("""
            COPY stg_vendas (
                id_venda,
                vendedor_id,
                estabelecimento,
                cnpj,
                rota,
                estado,
                municipio,
                pais,
                produto,
                tipo_embalagem,
                motivo_devolucao,
                quantidade,
                valor_venda,
                quantidade_devolucao,
                valor_devolucao,
                qtd_freezers,
                data_compra,
                vendedor,
                supervisor
            )
            FROM STDIN
            WITH CSV
            DELIMITER ";"
            NULL ""
        """, buffer)

        conn.commit()
        logging.info(f"Batch carregado: {i + 1} até {i + len(batch)}")


# ============================================================
# VALIDAÇÃO ANTES DE CONECTAR
# ============================================================

if not arquivo_sorvetes.exists():
    raise FileNotFoundError(
        f"Arquivo de vendas não encontrado: {arquivo_sorvetes}
"
        "Configure a variável de ambiente ARQUIVO_SORVETES com o caminho correto."
    )

if not arquivo_supervisores.exists():
    raise FileNotFoundError(
        f"Arquivo de supervisores não encontrado: {arquivo_supervisores}
"
        "Configure a variável de ambiente ARQUIVO_SUPERVISORES com o caminho correto."
    )

if not DB_CONFIG["password"]:
    raise EnvironmentError(
        "Senha do banco não configurada.
"
        "Configure a variável de ambiente DB_PASSWORD antes de rodar o ETL."
    )


# ============================================================
# CONEXÃO E PIPELINE
# ============================================================

conn = psycopg2.connect(
    host=DB_CONFIG["host"],
    dbname=DB_CONFIG["database"],
    user=DB_CONFIG["user"],
    password=DB_CONFIG["password"]
)

cursor = conn.cursor()

try:
    logging.info("Início do pipeline ETL")

    cursor.execute("SET datestyle TO 'ISO, DMY';")

    cursor.execute("ALTER TABLE fat_vendas DROP CONSTRAINT IF EXISTS fk_cliente;")
    cursor.execute("ALTER TABLE fat_vendas DROP CONSTRAINT IF EXISTS fk_localizacao;")
    cursor.execute("ALTER TABLE fat_vendas DROP CONSTRAINT IF EXISTS fk_vendedor;")
    cursor.execute("ALTER TABLE fat_vendas DROP CONSTRAINT IF EXISTS fk_produto;")
    cursor.execute("ALTER TABLE fat_vendas DROP CONSTRAINT IF EXISTS fk_tempo;")
    conn.commit()

    cursor.execute("TRUNCATE TABLE fat_vendas     RESTART IDENTITY CASCADE;")
    cursor.execute("TRUNCATE TABLE dim_cliente     RESTART IDENTITY CASCADE;")
    cursor.execute("TRUNCATE TABLE dim_localizacao RESTART IDENTITY CASCADE;")
    cursor.execute("TRUNCATE TABLE dim_vendedor    RESTART IDENTITY CASCADE;")
    cursor.execute("TRUNCATE TABLE dim_produto     RESTART IDENTITY CASCADE;")
    cursor.execute("TRUNCATE TABLE dim_tempo       RESTART IDENTITY CASCADE;")
    cursor.execute("TRUNCATE TABLE stg_vendas      RESTART IDENTITY CASCADE;")
    conn.commit()

    cursor.execute("""
        ALTER TABLE fat_vendas
        ADD COLUMN IF NOT EXISTS id_venda VARCHAR(50);
    """)

    cursor.execute("""
        ALTER TABLE fat_vendas
        ADD COLUMN IF NOT EXISTS quantidade_devolucao INT;
    """)

    cursor.execute("""
        CREATE UNIQUE INDEX IF NOT EXISTS uq_fat_vendas_id_venda
        ON fat_vendas (id_venda);
    """)
    conn.commit()

    df_sorvetes = pd.read_csv(
        arquivo_sorvetes,
        sep=",",
        encoding="utf-8-sig",
        dtype=str,
        keep_default_na=False
    )

    df_supervisores = pd.read_csv(
        arquivo_supervisores,
        sep=",",
        encoding="utf-8-sig",
        dtype=str,
        keep_default_na=False
    )

    df_sorvetes     = normalizar_colunas(df_sorvetes)
    df_supervisores = normalizar_colunas(df_supervisores)

    df_sorvetes = df_sorvetes.rename(columns={"nome_estabelecimento": "estabelecimento"})

    df_sorvetes     = df_sorvetes.loc[:,     ~df_sorvetes.columns.duplicated()]
    df_supervisores = df_supervisores.loc[:, ~df_supervisores.columns.duplicated()]

    df_sorvetes     = limpar_textos(df_sorvetes)
    df_supervisores = limpar_textos(df_supervisores)

    logging.info(f"Linhas base sorvetes: {len(df_sorvetes)}")
    logging.info(f"Linhas base supervisores: {len(df_supervisores)}")

    df_sorvetes["estado"] = "Alagoas"
    df_sorvetes["pais"]   = "Brasil"

    df_sorvetes["tipo_embalagem"] = (
        df_sorvetes["tipo_embalagem"]
        .replace("", "Avulso")
        .fillna("Avulso")
    )

    for col in ["quantidade_devolucao", "valor_devolucao_brl"]:
        df_sorvetes[col] = (
            df_sorvetes[col]
            .astype(str)
            .str.strip()
            .replace({"": "0", "nan": "0", "None": "0", "NULL": "0", "null": "0"})
            .fillna("0")
        )

    colunas_numericas = [
        "quantidade",
        "valor_compra_brl",
        "quantidade_devolucao",
        "valor_devolucao_brl",
        "qtd_freezers"
    ]

    for col in colunas_numericas:
        if col in df_sorvetes.columns:
            df_sorvetes[col] = tratar_numero_americano(df_sorvetes[col])

    for col in ["quantidade", "quantidade_devolucao", "qtd_freezers"]:
        if col in df_sorvetes.columns:
            df_sorvetes[col] = (
                pd.to_numeric(df_sorvetes[col], errors="coerce")
                .fillna(0)
                .astype(int)
            )

    df_sorvetes["data_compra"] = pd.to_datetime(
        df_sorvetes["data_compra"],
        errors="coerce",
        format="%Y-%m-%d"
    ).dt.strftime("%d/%m/%Y")

    datas_invalidas = df_sorvetes["data_compra"].isna().sum()
    logging.info(f"Datas inválidas após conversão: {datas_invalidas}")

    df_supervisores_aux = (
        df_supervisores[["vendedor_id", "supervisor"]]
        .drop_duplicates(subset=["vendedor_id"])
    )

    df_final = df_sorvetes.merge(df_supervisores_aux, on="vendedor_id", how="left")

    df_final = df_final.rename(columns={
        "valor_compra_brl":    "valor_venda",
        "valor_devolucao_brl": "valor_devolucao"
    })

    colunas_staging = [
        "id_venda", "vendedor_id", "estabelecimento", "cnpj", "rota",
        "estado", "municipio", "pais", "produto", "tipo_embalagem",
        "motivo_devolucao", "quantidade", "valor_venda",
        "quantidade_devolucao", "valor_devolucao", "qtd_freezers",
        "data_compra", "vendedor", "supervisor"
    ]

    colunas_faltantes = [col for col in colunas_staging if col not in df_final.columns]
    if colunas_faltantes:
        raise Exception(f"Colunas faltando no DataFrame: {colunas_faltantes}")

    df_staging = df_final[colunas_staging].copy()
    df_staging = df_staging.drop_duplicates(subset=["id_venda"], keep="last")
    df_staging = df_staging.where(pd.notnull(df_staging), "")

    logging.info(f"Linhas finais para staging: {len(df_staging)}")

    carregar_staging_em_batch(cursor, conn, df_staging)

    cursor.execute("""
        INSERT INTO dim_cliente (estabelecimento, cnpj, rota)
        SELECT DISTINCT ON (
            NULLIF(regexp_replace(cnpj, '\D', '', 'g'), '')::BIGINT
        )
            estabelecimento,
            NULLIF(regexp_replace(cnpj, '\D', '', 'g'), '')::BIGINT AS cnpj,
            rota
        FROM stg_vendas
        WHERE NULLIF(regexp_replace(cnpj, '\D', '', 'g'), '') IS NOT NULL
        ORDER BY
            NULLIF(regexp_replace(cnpj, '\D', '', 'g'), '')::BIGINT,
            estabelecimento
        ON CONFLICT (cnpj) DO NOTHING;
    """)

    cursor.execute("""
        INSERT INTO dim_localizacao (estado, municipio)
        SELECT DISTINCT estado, municipio
        FROM stg_vendas
        ON CONFLICT (estado, municipio) DO NOTHING;
    """)

    cursor.execute("""
        INSERT INTO dim_vendedor (vendedor, supervisor)
        SELECT DISTINCT vendedor, supervisor
        FROM stg_vendas
        ON CONFLICT (vendedor, supervisor) DO NOTHING;
    """)

    cursor.execute("""
        INSERT INTO dim_produto (produto, tipo_embalagem, motivo_devolucao)
        SELECT DISTINCT produto, tipo_embalagem, motivo_devolucao
        FROM stg_vendas
        ON CONFLICT (produto, tipo_embalagem, motivo_devolucao) DO NOTHING;
    """)

    cursor.execute("""
        INSERT INTO dim_tempo (
            data_compra, dia, mes, ano,
            bimestre, trimestre, semestre,
            mes_ordem, nome_dia, mes_ano, nome_mes
        )
        SELECT DISTINCT
            data_compra,
            EXTRACT(DAY     FROM data_compra)::INT,
            EXTRACT(MONTH   FROM data_compra)::INT,
            EXTRACT(YEAR    FROM data_compra)::INT,
            CEIL(EXTRACT(MONTH FROM data_compra) / 2.0)::INT,
            EXTRACT(QUARTER FROM data_compra)::INT,
            CASE WHEN EXTRACT(MONTH FROM data_compra) <= 6 THEN 1 ELSE 2 END,
            EXTRACT(MONTH FROM data_compra)::INT,
            INITCAP(TO_CHAR(data_compra, 'TMDay')),
            TO_CHAR(data_compra, 'MM/YYYY'),
            INITCAP(TO_CHAR(data_compra, 'TMMonth'))
        FROM stg_vendas
        WHERE data_compra IS NOT NULL
        ON CONFLICT (data_compra) DO NOTHING;
    """)

    conn.commit()
    logging.info("Dimensões carregadas")

    cursor.execute("""
        INSERT INTO fat_vendas (
            id_venda, id_cliente, id_localizacao, id_vendedor,
            id_produto, id_tempo, qtd_freezers, quantidade,
            quantidade_devolucao, valor_venda, valor_devolucao
        )
        SELECT
            s.id_venda,
            c.id_cliente,
            l.id_local,
            v.id_vendedor,
            p.id_produto,
            t.id_tempo,
            s.qtd_freezers,
            s.quantidade,
            s.quantidade_devolucao,
            s.valor_venda,
            s.valor_devolucao
        FROM stg_vendas s

        LEFT JOIN dim_cliente c
            ON c.cnpj = NULLIF(regexp_replace(s.cnpj, '\D', '', 'g'), '')::BIGINT

        LEFT JOIN dim_localizacao l
            ON l.estado    = s.estado
           AND l.municipio = s.municipio

        LEFT JOIN dim_vendedor v
            ON v.vendedor                = s.vendedor
           AND COALESCE(v.supervisor, '') = COALESCE(s.supervisor, '')

        LEFT JOIN dim_produto p
            ON p.produto        = s.produto
           AND p.tipo_embalagem = s.tipo_embalagem
           AND COALESCE(p.motivo_devolucao, '') = COALESCE(s.motivo_devolucao, '')

        LEFT JOIN dim_tempo t
            ON t.data_compra = s.data_compra

        ON CONFLICT (id_venda) DO UPDATE SET
            id_cliente           = EXCLUDED.id_cliente,
            id_localizacao       = EXCLUDED.id_localizacao,
            id_vendedor          = EXCLUDED.id_vendedor,
            id_produto           = EXCLUDED.id_produto,
            id_tempo             = EXCLUDED.id_tempo,
            qtd_freezers         = EXCLUDED.qtd_freezers,
            quantidade           = EXCLUDED.quantidade,
            quantidade_devolucao = EXCLUDED.quantidade_devolucao,
            valor_venda          = EXCLUDED.valor_venda,
            valor_devolucao      = EXCLUDED.valor_devolucao;
    """)

    conn.commit()
    logging.info("Fato carregada")

    cursor.execute("""
        ALTER TABLE fat_vendas
        ADD CONSTRAINT fk_cliente
        FOREIGN KEY (id_cliente) REFERENCES dim_cliente(id_cliente);
    """)

    cursor.execute("""
        ALTER TABLE fat_vendas
        ADD CONSTRAINT fk_localizacao
        FOREIGN KEY (id_localizacao) REFERENCES dim_localizacao(id_local);
    """)

    cursor.execute("""
        ALTER TABLE fat_vendas
        ADD CONSTRAINT fk_vendedor
        FOREIGN KEY (id_vendedor) REFERENCES dim_vendedor(id_vendedor);
    """)

    cursor.execute("""
        ALTER TABLE fat_vendas
        ADD CONSTRAINT fk_produto
        FOREIGN KEY (id_produto) REFERENCES dim_produto(id_produto);
    """)

    cursor.execute("""
        ALTER TABLE fat_vendas
        ADD CONSTRAINT fk_tempo
        FOREIGN KEY (id_tempo) REFERENCES dim_tempo(id_tempo);
    """)

    conn.commit()
    logging.info("Foreign keys recriadas")
    logging.info("Pipeline ETL finalizado com sucesso")
    print("ETL completo executado com sucesso!")

except Exception as e:
    conn.rollback()
    logging.exception(f"Erro no pipeline: {e}")
    print(f"Erro no pipeline: {e}")
    raise

finally:
    cursor.close()
    conn.close()
